# 소나타 2026 DB Chunk TF-IDF 분석

현재 PostgreSQL에 적재된 소나타 chunk를 사용한다. PDF 재처리나 임시 chunk 생성은 하지 않는다.

## 분석 목적
DB corpus에 TF-IDF 검색을 실행하고 기존 CarManualSearchService의 embedding/pgvector 검색 결과를 같은 질문으로 비교한다.

In [1]:
import os
from dotenv import load_dotenv
import psycopg
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
load_dotenv()
# DB_URL 비밀값을 출력하지 않는다.
connection = psycopg.connect(os.environ["DB_URL"])
cursor = connection.cursor()


## Sonata 차량 확인

In [2]:
cursor.execute(
    """SELECT car_id,car_brand_nm,car_brand_eng_nm,car_nm,car_eng_nm,car_model_yr
       FROM public.car
       WHERE car_brand_eng_nm=%s AND car_eng_nm=%s AND car_model_yr=%s""",
    ("hyundai","sonata",2026),
)
cars = cursor.fetchall()
assert len(cars) == 1
car_id = cars[0][0]
print(cars)


[('20261003_000014', '현대', 'hyundai', '소나타', 'sonata', 2026)]


## 실제 DB chunk 조회와 기초 통계
차량 결과에서 확인한 CAR_ID만 사용한다.

In [3]:
cursor.execute(
    """SELECT h.car_manual_chapter_no,h.car_manual_chapter_nm,
       d.car_manual_chapter_id,d.car_manual_chunk_id,d.car_manual_chunk_page_no,
       d.car_manual_chunk_no,d.car_manual_chunk_txt
       FROM public.car_manual_chunk d JOIN public.car_manual_chapter h
       ON h.car_id=d.car_id AND h.car_manual_chapter_id=d.car_manual_chapter_id
       WHERE d.car_id=%s ORDER BY d.car_manual_chunk_no""",(car_id,))
rows=cursor.fetchall()
cursor.execute(
    """SELECT COUNT(*),COUNT(*) FILTER(WHERE car_manual_chunk_txt IS NULL),
       COUNT(*) FILTER(WHERE car_manual_chunk_txt=''),
       COUNT(*) FILTER(WHERE car_manual_chunk_txt IS NULL OR btrim(car_manual_chunk_txt)=''),
       COUNT(*) FILTER(WHERE car_manual_chunk_embed_vec IS NULL),
       COUNT(*) FILTER(WHERE car_manual_chunk_page_no IS NULL),
       COUNT(car_manual_chunk_txt)-COUNT(DISTINCT car_manual_chunk_txt),
       AVG(char_length(car_manual_chunk_txt)),
       percentile_cont(0.5) WITHIN GROUP(ORDER BY char_length(car_manual_chunk_txt)),
       MIN(char_length(car_manual_chunk_txt)),MAX(char_length(car_manual_chunk_txt))
       FROM public.car_manual_chunk WHERE car_id=%s""",(car_id,))
stats=cursor.fetchone()
cursor.execute("SELECT COUNT(*) FROM public.car_manual_chapter WHERE car_id=%s",(car_id,))
chapter_count=cursor.fetchone()[0]
print("chapters:",chapter_count)
print("chunks,NULL,empty,blank,NULL_embedding,NULL_page,duplicate_rows,mean,median,min,max:",stats)


chapters: 10
chunks,NULL,empty,blank,NULL_embedding,NULL_page,duplicate_rows,mean,median,min,max: (947, 0, 0, 0, 0, 0, 0, Decimal('607.3495248152059134'), 717.0, 19, 800)


## Corpus와 TF-IDF
한국어 형태소 분석기 없이 문자 2~4gram을 사용한다.

In [4]:
corpus_rows=[r for r in rows if r[6] is not None and r[6].strip()]
corpus=[r[6] for r in corpus_rows]
vectorizer=TfidfVectorizer(analyzer="char",ngram_range=(2,4),min_df=1,max_df=1.0,lowercase=True,sublinear_tf=True)
tfidf_matrix=vectorizer.fit_transform(corpus)
print("documents:",len(corpus),"vocabulary:",len(vectorizer.vocabulary_))


documents: 947 vocabulary: 175451


## 근거 chunk를 확인한 질문

In [5]:
questions=[
 "가솔린/LPI 엔진에 권장되는 엔진오일 등급과 용량은?",
 "후드는 어떻게 안전하게 닫나요?",
 "배터리 단자와 윗부분은 어떻게 관리하나요?",
]
targets={questions[0]:(23,39),questions[1]:(164,317),questions[2]:(475,885)}
for q,target in targets.items():
    r=next(row for row in rows if (row[4],row[5])==target)
    print(q,"=> page",r[4],"chapter",r[1],"chunk",r[5],r[6][:150].replace("\\n"," "))


가솔린/LPI 엔진에 권장되는 엔진오일 등급과 용량은? => page 23 chapter 1. 안내 및 차량 정보 chunk 39 1
1-17
추천 오일 및 용량
가솔린/LPI 엔진
*1:엔진 오일 용량은 일반적인 오일 교체 시 주입되는 용량 기준입니다. 
*2:API SN PLUS(또는 그 이상) FULL Synthetic 등급의 엔진오일 사용을 추천해드립니다. 만약 더 낮은 등급의 엔진오일(광
후드는 어떻게 안전하게 닫나요? => page 164 chapter 5. 편의 장치 chunk 317 편의 장치
5-56
후드 닫는 방법
1. 후드를 닫기 전에 다음 사항을 먼저 확인하
십시오.
• 엔진룸의 모든 주입구 캡이 정확하게 닫혀
있는지 확인하십시오.
• 엔진룸에 불필요한 물건은 없는지 확인하
십시오(장갑, 공구 등).
2. 후드를 내린 후 아래로 밀어 닫으십
배터리 단자와 윗부분은 어떻게 관리하나요? => page 475 chapter 9. 정기 점검 chunk 885 9
9-29
배터리
배터리 관리
2C_TakingCareOfBattery
• 배터리는 수시로 점검하고, 항상 깨끗하게
유지 하십시오. 오염된 채로 방치하면 배터
리의 수명이 단축됩니다.
• 배터리를 재충전하거나 점검을 실시하기 전
에, 엔진을 끄고 모든 전기 장치를 끄


## TF-IDF 검색 top-5
점수는 cosine similarity이며 page/chapter/chunk_no는 DB metadata다.

In [6]:
def tfidf_top5(q):
    scores=cosine_similarity(vectorizer.transform([q]),tfidf_matrix).ravel()
    return [(i,float(scores[i]),corpus_rows[int(i)]) for i in scores.argsort()[::-1][:5]]
for q in questions:
    print(q)
    for rank,(i,score,r) in enumerate(tfidf_top5(q),1):
        print(rank,round(score,6),r[4],r[1],r[5],r[6][:100].replace("\\n"," "))


가솔린/LPI 엔진에 권장되는 엔진오일 등급과 용량은?
1 0.212696 7 1. 안내 및 차량 정보 9 타이어 에너지 소비 효율 등급 ...................................................................................
2 0.179207 23 1. 안내 및 차량 정보 39 1
1-17
추천 오일 및 용량
가솔린/LPI 엔진
*1:엔진 오일 용량은 일반적인 오일 교체 시 주입되는 용량 기준입니다. 
*2:API SN PLUS(또는 그 이상) FULL 
3 0.171594 7 1. 안내 및 차량 정보 10 가솔린/LPI 엔진..........................................................................................
4 0.082588 506 색인 944 고장 자동차 정지 표지판 (안전 삼각대) 8-3
교차로/건널목에서 시동이 꺼진 경우 8-2
브레이크 제동력이 좋지 않을 경우 8-3
주행 중 시동이 꺼진 경우 8-3
주행 중 차량
5 0.072437 24 1. 안내 및 차량 정보 41 안내 및 차량 정보
1-18
 ઱੄
 
• 각 시스템의 원활한 작동과 수명을 위해 품질과 성능이 적합한 부품(엔진 오일, 변속기 오일 등)
을 사용하십시오. 품질이나 성능이 부적합
후드는 어떻게 안전하게 닫나요?
1 0.07648 181 5. 편의 장치 345 5
5-73
SD 메모리
2C_BuiltInCamSDMemory
녹화 영상 저장을 위한 SD 메모리의 수명 상태,
SD 메모리 단자 위치를 확인할 수 있습니다. 
SD 메모리를 안
2 0.067624 282 6. 시동 및 주행 525 우 배터리의 전압이 회복된 후 주행하십시
오. 배터리 전압이 낮으면 엔진 회전이 불안
정하고 ABS 경고등이 켜지거나 깜일 수 있
습니다. 
• 타이어 주위를 청소할 때는 각 휠에
3 0.056168 423 7. 운전자 보조 794 • 원격 스마트 주차 보조는

## 현재 Embedding 검색 top-5
기존 CarManualSearchService 검색만 호출하며 LLM 답변 chain은 실행하지 않는다.

In [7]:
from car_search_rag.common.sql_session import SqlSession
from car_search_rag.car_search.car_manual_search_service import CarManualSearchService
service=CarManualSearchService(sql_session=SqlSession(result_log=False))
embedding_results={q:service.search_manual("hyundai","sonata",2026,q,limit=5) for q in questions}
for q in questions:
    print(q)
    for rank,d in enumerate(embedding_results[q],1):
        print(rank,d["similarity"],d["carManualChunkPageNo"],d["carManualChapterNm"],d["carManualChunkNo"])


가솔린/LPI 엔진에 권장되는 엔진오일 등급과 용량은?
1 0.667742859480001 7 1. 안내 및 차량 정보 10
2 0.59359797795138 23 1. 안내 및 차량 정보 39
3 0.486773260942454 24 1. 안내 및 차량 정보 41
4 0.486773260942454 24 1. 안내 및 차량 정보 41
5 0.486271662347336 464 9. 정기 점검 868
후드는 어떻게 안전하게 닫나요?
1 0.552632820694804 110 5. 편의 장치 201
2 0.48009150844453 70 3. 시트 및 안전 장치 127
3 0.47080782473408 61 3. 시트 및 안전 장치 112
4 0.457098854767398 164 5. 편의 장치 317
5 0.454021485372144 333 7. 운전자 보조 634
배터리 단자와 윗부분은 어떻게 관리하나요?
1 0.469004349732316 475 9. 정기 점검 885
2 0.438865135773312 448 9. 정기 점검 844
3 0.397609120330958 233 5. 편의 장치 425
4 0.397609120330958 233 5. 편의 장치 425
5 0.383719567918538 502 색인 934


## 동일 질문 비교, 한계, 결론
지정한 근거 chunk가 top-5에 포함되는지 확인한다.

In [8]:
for q in questions:
    page,chunk=targets[q]
    thit=any(r[4]==page and r[5]==chunk for _,_,r in tfidf_top5(q))
    ehit=any(int(d["carManualChunkPageNo"])==page and int(d["carManualChunkNo"])==chunk for d in embedding_results[q])
    print(q,"TF-IDF:",thit,"embedding:",ehit)


가솔린/LPI 엔진에 권장되는 엔진오일 등급과 용량은? TF-IDF: True embedding: True
후드는 어떻게 안전하게 닫나요? TF-IDF: False embedding: True
배터리 단자와 윗부분은 어떻게 관리하나요? TF-IDF: True embedding: True


세 질문에서 TF-IDF는 두 근거를 찾고 후드 근거를 놓쳤다. embedding 검색은 지정 근거 세 개를 모두 top-5에서 반환했다. 세 질문뿐인 예시이므로 전체 검색 품질이나 통계적 우열이 아니다. 이미지 join으로 동일 chunk 중복 반환도 관찰됐다. PDF 원본 페이지 통계는 DB 결과만으로 추론하지 않는다.